#DATA CLEANING — Employee Attrition Risk Dashboard

Dataset : Employee Performance Data Set (employee_performance_pro.csv, 500 baris)

Output  : employees_clean.csv  (dihubungkan ke Looker Studio)


In [ ]:
# Menghubungkan ke Google Drive
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Konfigurasi
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/Project/Employee-Attrition-Risk-Dashboard ")

In [ ]:
INPUT_FILE  = "employee_performance_pro.csv"   # nama file mentah di dalam BASE_DIR
OUTPUT_FILE = "employees_clean.csv"            # nama file hasil

In [ ]:
INPUT_PATH  = BASE_DIR / INPUT_FILE
OUTPUT_PATH = BASE_DIR / OUTPUT_FILE

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan: {INPUT_PATH}\n"
        f"Isi folder {BASE_DIR}: {[p.name for p in BASE_DIR.glob('*')] if BASE_DIR.exists() else 'folder tidak ada'}")

REFERENCE_YEAR = 2025   # tahun acuan: YearsAtCompany di dataset = 2025 - tahun bergabung
KEEP_PHONE     = False  # False = kolom nomor telepon tidak diekspor (data pribadi, tidak dipakai di dashboard)

In [ ]:
# Lihat data
df = pd.read_csv(INPUT_PATH)
print("Ukuran data:", df.shape)
print(df.dtypes)
df.head()


Ukuran data: (500, 24)
EmployeeID                 int64
Name                      object
Gender                    object
Age                        int64
Department                object
JobRole                   object
EducationLevel             int64
JoiningDate               object
CountryCode                int64
Country                   object
PhoneNumber                int64
MonthlySalary              int64
OvertimeHoursPerMonth      int64
LeavesTaken                int64
LastLeaveDate             object
LeaveDayName              object
ProjectsHandled            int64
TrainingHours              int64
CustomerSatisfaction     float64
LastPromotionYear          int64
YearsAtCompany             int64
WorkLifeBalanceScore     float64
PerformanceRating          int64
AttritionRisk             object
dtype: object


,EmployeeID,Name,Gender,Age,Department,JobRole,EducationLevel,JoiningDate,CountryCode,Country,...,LastLeaveDate,LeaveDayName,ProjectsHandled,TrainingHours,CustomerSatisfaction,LastPromotionYear,YearsAtCompany,WorkLifeBalanceScore,PerformanceRating,AttritionRisk
0,1,Steven Barnett,Other,57,Finance,Auditor,2,2016-05-05,91,India,...,2024-07-03,Wednesday,13,16,NaN,2020,9,1.90,4,No
1,2,Christopher Benson,Female,26,Sales,Sales Executive,2,2014-08-20,1,Canada,...,2024-01-05,Friday,15,44,8.0,2014,11,5.03,1,Yes
2,3,Norman Lane,Other,59,Support,Helpdesk,1,2010-05-17,1,Canada,...,2024-11-27,Wednesday,7,62,7.0,2012,15,4.83,4,No
3,4,Rita Walker,Female,43,HR,HR Executive,3,2015-06-20,49,Germany,...,2024-07-15,Monday,15,8,NaN,2016,10,4.60,4,No
4,5,Judith Ware,Male,52,Sales,Account Manager,3,2019-08-20,49,Germany,...,2024-12-17,Tuesday,10,57,8.0,2022,6,3.73,4,No


Mengaudit Kualitas Data - sebelum Data Cleaning

---

In [ ]:
print("=== Missing value ===")
na = df.isna().sum()
print(na[na > 0] if na.any() else "Tidak ada")

=== Missing value ===
CustomerSatisfaction    319
dtype: int64


In [ ]:
print("\n=== Duplikat ===")
print("Baris duplikat penuh :", df.duplicated().sum())
print("EmployeeID duplikat  :", df["EmployeeID"].duplicated().sum())


=== Duplikat ===
Baris duplikat penuh : 0
EmployeeID duplikat  : 0


In [ ]:
print("\n=== Cek rentang nilai ===")
checks = {
    "Age di luar 18-65"                  : ~df["Age"].between(18, 65),
    "PerformanceRating di luar 1-5"      : ~df["PerformanceRating"].between(1, 5),
    "EducationLevel di luar 1-5"         : ~df["EducationLevel"].between(1, 5),
    "CustomerSatisfaction di luar 1-10"  : df["CustomerSatisfaction"].notna() & ~df["CustomerSatisfaction"].between(1, 10),
    "Overtime / Leaves / Training < 0"   : (df[["OvertimeHoursPerMonth", "LeavesTaken", "TrainingHours"]] < 0).any(axis=1),
    "MonthlySalary <= 0"                 : df["MonthlySalary"] <= 0,
    "WorkLifeBalanceScore < 0"           : df["WorkLifeBalanceScore"] < 0,
}
for name, mask in checks.items():
    print(f"{name:38s}: {int(mask.sum())} baris")


=== Cek rentang nilai ===
Age di luar 18-65                     : 0 baris
PerformanceRating di luar 1-5         : 0 baris
EducationLevel di luar 1-5            : 0 baris
CustomerSatisfaction di luar 1-10     : 0 baris
Overtime / Leaves / Training < 0      : 0 baris
MonthlySalary <= 0                    : 0 baris
WorkLifeBalanceScore < 0              : 56 baris


In [ ]:
print("\n=== Cek konsistensi ===")
join = pd.to_datetime(df["JoiningDate"], errors="coerce")
leave = pd.to_datetime(df["LastLeaveDate"], errors="coerce")
print("Tanggal tidak terbaca (join/leave)   :", join.isna().sum(), "/", leave.isna().sum())
print("Masa kerja tidak cocok dg tahun join :", (df["YearsAtCompany"] != REFERENCE_YEAR - join.dt.year).sum())
print("Promosi sebelum tahun bergabung      :", (df["LastPromotionYear"] < join.dt.year).sum())
print("Nama hari cuti tidak cocok tanggal   :", (leave.dt.day_name() != df["LeaveDayName"]).sum())
print("Bergabung sebelum usia 18            :", ((df["Age"] - df["YearsAtCompany"]) < 18).sum())
print("Nomor telepon bukan 10 digit         :", (df["PhoneNumber"].astype(str).str.len() != 10).sum())
print("CustomerSatisfaction kosong per dept :")
print(df.groupby("Department")["CustomerSatisfaction"].apply(lambda s: s.isna().mean()).round(2).to_dict())


=== Cek konsistensi ===
Tanggal tidak terbaca (join/leave)   : 0 / 0
Masa kerja tidak cocok dg tahun join : 0
Promosi sebelum tahun bergabung      : 0
Nama hari cuti tidak cocok tanggal   : 0
Bergabung sebelum usia 18            : 73
Nomor telepon bukan 10 digit         : 56
CustomerSatisfaction kosong per dept :
{'Finance': 1.0, 'HR': 1.0, 'IT': 1.0, 'Marketing': 1.0, 'Sales': 0.0, 'Support': 0.0}


Data Cleaning

---



In [ ]:
# Merapikan nama kolom dan tipe data
def to_snake(c):
    import re
    c = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", c.strip())
    return c.lower()

df.columns = [to_snake(c) for c in df.columns]
df = df.rename(columns={"employee_i_d": "employee_id", "name": "employee_name"})
print(list(df.columns))

df["joining_date"]    = pd.to_datetime(df["joining_date"], errors="coerce").dt.normalize()
df["last_leave_date"] = pd.to_datetime(df["last_leave_date"], errors="coerce").dt.normalize()

for col in ["employee_name", "gender", "department", "job_role", "country", "leave_day_name", "attrition_risk"]:
    df[col] = df[col].astype(str).str.strip()

df = df.drop_duplicates()
df = df.dropna(subset=["employee_id", "department", "performance_rating", "joining_date"])
print("Ukuran setelah cleaning dasar:", df.shape)

['employee_id', 'employee_name', 'gender', 'age', 'department', 'job_role', 'education_level', 'joining_date', 'country_code', 'country', 'phone_number', 'monthly_salary', 'overtime_hours_per_month', 'leaves_taken', 'last_leave_date', 'leave_day_name', 'projects_handled', 'training_hours', 'customer_satisfaction', 'last_promotion_year', 'years_at_company', 'work_life_balance_score', 'performance_rating', 'attrition_risk']
Ukuran setelah cleaning dasar: (500, 24)


In [ ]:
# Memperbaiki nomor telepon
# PhoneNumber terbaca sebagai angka sehingga angka 0 di depan bisa hilang
# (di data ini ada nomor yang hanya 8-9 digit).
#Solusi: Ubah ke teks, beri flag, gabung dengan kode negara.
phone_raw = df["phone_number"].astype(str).str.replace(r"\D", "", regex=True)
df["phone_length_issue"] = (phone_raw.str.len() != 10).astype(int)
df["phone_number"] = phone_raw.str.zfill(10)
df["phone_full"] = "+" + df["country_code"].astype(str) + " " + df["phone_number"]

In [ ]:
# Menangani nilai kosong dan data tidak wajar
# 7a. CustomerSatisfaction: kosong karena hanya diisi untuk Sales & Support, bukan data hilang.
#     Solusi: Jangan diisi median. Biarkan kosong dan tambah penanda peran yang berhadapan dengan klien.
df["is_client_facing"] = df["department"].isin(["Sales", "Support"]).astype(int)

In [ ]:
# 7b. Karyawan yang bergabung sebelum usia 18 -> beri flag untuk ditinjau, jangan dihilangkan.
df["joined_before_18_flag"] = ((df["age"] - df["years_at_company"]) < 18).astype(int)

In [ ]:
# 7c. WorkLifeBalanceScore negatif (turunan dari overtime dan cuti) -> flag, nilai asli dipertahankan.
df["wlb_negative_flag"] = (df["work_life_balance_score"] < 0).astype(int)

print("Flag usia   :", df["joined_before_18_flag"].sum())
print("Flag WLB<0  :", df["wlb_negative_flag"].sum())
print("Flag telepon:", df["phone_length_issue"].sum())

Flag usia   : 73
Flag WLB<0  : 56
Flag telepon: 56


In [ ]:
# Membuat kolom baru (feature engineering)
# 8a. Label teks
edu_map = {1: "1 - High School", 2: "2 - Diploma", 3: "3 - Bachelor", 4: "4 - Master", 5: "5 - Doctorate"}
# Alert: dataset hanya menjelaskan level 1 (High School) dan 5 (Doctorate). Label 2-4 asumsi saja.
df["education_label"] = df["education_level"].map(edu_map)

perf_map = {1: "1 - Sangat Rendah", 2: "2 - Rendah", 3: "3 - Cukup", 4: "4 - Baik", 5: "5 - Sangat Baik"}
df["performance_label"] = df["performance_rating"].map(perf_map)

In [ ]:
# 8b. Atribut waktu
df["join_year"] = df["joining_date"].dt.year
df["leave_month"] = df["last_leave_date"].dt.month
df["leave_month_name"] = df["last_leave_date"].dt.strftime("%m - %b")
df["leave_day_order"] = df["last_leave_date"].dt.dayofweek + 1             # 1 = Senin
df["years_since_promotion"] = REFERENCE_YEAR - df["last_promotion_year"]

In [ ]:
# 8c. Bucket. Label diawali angka/nol agar urutan di Looker Studio benar
def bucket(series, bins, labels):
    return pd.cut(series, bins=bins, labels=labels).astype(str)

df["overtime_bucket"] = bucket(df["overtime_hours_per_month"], [-1, 10, 20, 30, 40],
                               ["00-10 jam", "11-20 jam", "21-30 jam", "31-40 jam"])
df["wlb_bucket"] = bucket(df["work_life_balance_score"], [-np.inf, 2, 4, 6, np.inf],
                          ["1. Rendah (<2)", "2. Sedang (2-4)", "3. Cukup (4-6)", "4. Baik (>=6)"])
df["leaves_bucket"] = bucket(df["leaves_taken"], [-1, 10, 20, 30],
                             ["00-10 hari", "11-20 hari", "21-30 hari"])
df["training_bucket"] = bucket(df["training_hours"], [-1, 20, 40, 60, 80],
                               ["00-20 jam", "21-40 jam", "41-60 jam", "61-80 jam"])
df["tenure_bucket"] = bucket(df["years_at_company"], [-1, 3, 7, 11, 15],
                             ["00-03 thn", "04-07 thn", "08-11 thn", "12-15 thn"])
df["age_group"] = bucket(df["age"], [0, 29, 39, 49, 100],
                         ["20-29", "30-39", "40-49", "50-60"])

In [ ]:
# Kolom risiko (inti dashboard)
df["at_risk_flag"] = (df["attrition_risk"].str.lower() == "yes").astype(int)   # AVG(at_risk_flag) = risk rate
df["risk_status"]  = np.where(df["at_risk_flag"] == 1, "At Risk", "Not At Risk")

df["high_overtime_flag"] = (df["overtime_hours_per_month"] > 20).astype(int)
df["low_performance_flag"] = (df["performance_rating"] <= 2).astype(int)

In [ ]:
# Segmen: dari data terlihat risiko hanya muncul jika performa rendah dan overtime tinggi.
df["risk_segment"] = np.select(
    [
        (df["at_risk_flag"] == 1),
        (df["high_overtime_flag"] == 1) & (df["low_performance_flag"] == 0),
        (df["low_performance_flag"] == 1) & (df["high_overtime_flag"] == 0),
    ],
    ["1. At Risk (performa rendah + overtime tinggi)",
     "2. Watchlist (overtime tinggi, performa baik)",
     "3. Performa rendah, overtime normal"],
    default="4. Aman")
print(df["risk_segment"].value_counts().sort_index())

risk_segment
1. At Risk (performa rendah + overtime tinggi)     55
2. Watchlist (overtime tinggi, performa baik)     188
3. Performa rendah, overtime normal                30
4. Aman                                           227
Name: count, dtype: int64


In [ ]:
# Validasi akhir
assert df["employee_id"].is_unique, "EmployeeID masih duplikat"
assert df["at_risk_flag"].isin([0, 1]).all()
assert df["performance_rating"].between(1, 5).all()
assert df["joining_date"].notna().all() and df["last_leave_date"].notna().all()
assert df.drop(columns="customer_satisfaction").isna().sum().sum() == 0, "Ada missing selain customer_satisfaction"
# Notes: customer_satisfaction boleh kosong, tetapi hanya di departemen non-client-facing
assert df.loc[df["is_client_facing"] == 1, "customer_satisfaction"].notna().all()
assert df.loc[df["is_client_facing"] == 0, "customer_satisfaction"].isna().all()

print("Jumlah baris   :", len(df))
print("Jumlah kolom   :", df.shape[1])
print("At Risk        :", df["at_risk_flag"].sum(), f"({df['at_risk_flag'].mean():.1%})")
print("Rentang join   :", df["joining_date"].min().date(), "->", df["joining_date"].max().date())
print("Rentang cuti   :", df["last_leave_date"].min().date(), "->", df["last_leave_date"].max().date())

Jumlah baris   : 500
Jumlah kolom   : 47
At Risk        : 55 (11.0%)
Rentang join   : 2010-01-21 -> 2023-12-23
Rentang cuti   : 2024-01-01 -> 2024-12-28


In [ ]:
# Ekspor ke Google Drive
export = df.copy()
if not KEEP_PHONE:
    export = export.drop(columns=["phone_number", "phone_full"])
export["joining_date"] = export["joining_date"].dt.strftime("%Y-%m-%d")        # format ISO agar terbaca sebagai Date
export["last_leave_date"] = export["last_leave_date"].dt.strftime("%Y-%m-%d")
BASE_DIR.mkdir(parents=True, exist_ok=True)
export.to_csv(OUTPUT_PATH, index=False)
print("Tersimpan di:", OUTPUT_PATH, "| kolom:", export.shape[1])

Tersimpan di: /content/drive/MyDrive/Project/Employee-Attrition-Risk-Dashboard /employees_clean.csv | kolom: 45


In [ ]:
# Angka ringkas untuk teks insight dashboard
def rate(col):
    return (df.groupby(col)["at_risk_flag"].agg(jumlah_berisiko="sum", total="count", risk_rate="mean")
              .assign(risk_rate=lambda x: (x["risk_rate"] * 100).round(1)))

for col in ["department", "country", "gender", "overtime_bucket", "wlb_bucket", "performance_rating", "tenure_bucket"]:
    print(f"\n--- Risk rate per {col} ---")
    print(rate(col))

print("\nRata-rata customer satisfaction per departemen (Sales & Support saja):")
print(df.groupby("department")["customer_satisfaction"].mean().dropna().round(2))


--- Risk rate per department ---
            jumlah_berisiko  total  risk_rate
department                                   
Finance                  11     77       14.3
HR                        8     77       10.4
IT                       16     93       17.2
Marketing                 3     72        4.2
Sales                    11     99       11.1
Support                   6     82        7.3

--- Risk rate per country ---
         jumlah_berisiko  total  risk_rate
country                                   
Canada                10     92       10.9
France                10     80       12.5
Germany                8     77       10.4
India                  9     90       10.0
UK                     8     85        9.4
USA                   10     76       13.2

--- Risk rate per gender ---
        jumlah_berisiko  total  risk_rate
gender                                   
Female               13    186        7.0
Male                 18    165       10.9
Other                24  